# Procurement Intelligence Assistant — MVP

RAG-based chatbot answering procurement, sourcing, and supply chain questions
from a curated set of YouTube video transcripts.

*Pipeline stages:* transcription (done) → chunking → embeddings → vector store → retrieval + LLM

## 1. Setup

In [1]:
# Core dependencies for chunking and token counting
import json
import tiktoken

# cl100k_base is the tokenizer used by GPT-3.5/4 family models —
# a reasonable proxy for token count even if using a different embedding model
enc = tiktoken.get_encoding("cl100k_base")
def count_tokens(text: str) ->int:
    return len(enc.encode(text))

## 2. Load Transcripts


#Load transcripts

Sourcedata: data/transcripts.json
Structure: {video_id: {title, segments: [{start, end, text}, ...]}}
#Two videos (uSjrTpJHn2g, U1E6rtnreoA) were pre-trimmed to remove webinar preamble/roll-call content before this file was saved.

In [2]:
with open("data/transcripts.json") as f:
    data = json.load(f)

print(f"Loaded {len(data)} videos")

Loaded 10 videos


## 3. Chunking

*Strategy:* merge consecutive Whisper segments into ~250-token chunks,
with ~40-token overlap so ideas that span a chunk boundary (e.g. a numbered
list) still appear complete in at least one chunk.

Each chunk keeps video_id, title, start, end as metadata —
needed later for citations and clickable timestamped YouTube links.

In [3]:
def count_tokens(text: str) -> int:
    """Return the token count of a string using the cl100k_base tokenizer."""
    return len(enc.encode(text))


def chunk_segments(segments, target_tokens=250, overlap_tokens=40):
    """
    Merge a list of {start, end, text} segments into token-bounded chunks.

    Args:
        segments: list of transcript segments for one video
        target_tokens: approx. token count to accumulate before closing a chunk
        overlap_tokens: approx. token count carried over into the next chunk

    Returns:
        list of {text, start, end, token_count} dicts
    """
    chunks = []
    current = []
    current_tokens = 0

    for seg in segments:
        current.append(seg)
        current_tokens += count_tokens(seg["text"])

        if current_tokens >= target_tokens:
            chunks.append(_build_chunk(current, current_tokens))
            current, current_tokens = _take_overlap(current, overlap_tokens)

    if current:
        chunks.append(_build_chunk(current, current_tokens))

    return chunks


def _build_chunk(seg_list, token_count):
    """Join a list of segments into a single chunk record."""
    return {
        "text": " ".join(s["text"].strip() for s in seg_list),
        "start": seg_list[0]["start"],
        "end": seg_list[-1]["end"],
        "token_count": token_count,
    }


def _take_overlap(seg_list, overlap_tokens):
    """Return the trailing segments (and their token count) to seed the next chunk."""
    overlap_seg_list = []
    overlap_count = 0
    for s in reversed(seg_list):
        overlap_count += count_tokens(s["text"])
        overlap_seg_list.insert(0, s)
        if overlap_count >= overlap_tokens:
            break
    return overlap_seg_list, overlap_count

## 4.Test chunk size in one video

In [4]:
test_video = "ZCIJQJRw6xw"

for target in [200, 250, 300]:
    test_chunks = chunk_segments(
        data[test_video]["segments"],
        target_tokens=target,
        overlap_tokens=int(target * 0.15)
    )
    print(f"\n=== target={target} tokens ({len(test_chunks)} chunks) ===")
    print(test_chunks[1]["text"])


=== target=200 tokens (13 chunks) ===
Whether it's software for a startup or machines for a factory, procurement decides who delivers what, when, and how much it costs. For example, think of procurement like planning a big wedding. You don't just go buy food and flowers. You plan your guest list, find reliable caterers, compare quotations, ensure everything arrives on time, track the budget, sign contracts. That's procurement on a business scale. Let's take a real-world example. Tata Motors, one of India's largest automobile manufacturers. To build each vehicle, they need tires from Bridgestone, steel from Tata Steel, electronics from Bosch, paint, seats, dashboards, software systems, etc. Tata Motors doesn't manufacture all of this in-house. Instead, they procure parts from a global network of suppliers who specialize in those products. If even one part, like a microchip, doesn't arrive on time, production halts, deadlines are missed, and millions are lost. That's how powerful procur

## 5. Lock in final size and run on all videos

In [5]:
FINAL_TARGET = 250   # <- update based on Step 4
FINAL_OVERLAP = 40    # <- update based on Step 4

all_chunks = []

for vid, content in data.items():
    video_chunks = chunk_segments(
        content["segments"],
        target_tokens=FINAL_TARGET,
        overlap_tokens=FINAL_OVERLAP
    )
    for c in video_chunks:
        c["video_id"] = vid
        c["title"] = content["title"]
        all_chunks.append(c)

print(f"Total chunks: {len(all_chunks)}")
print(f"Videos processed: {len(data)}")
print(f"Avg chunks per video: {len(all_chunks) / len(data):.1f}")

Total chunks: 160
Videos processed: 10
Avg chunks per video: 16.0


## 6. Sanity check the full set

In [6]:
import random

sample_chunks = random.sample(all_chunks, 5)

for c in sample_chunks:
    print(f"--- {c['title']} ({c['start']:.0f}s-{c['end']:.0f}s, {c['token_count']} tokens) ---")
    print(c["text"])
    print()

--- 6-Step Strategic Sourcing Flywheel: Turning Supplier Relationships into Results (2597s-2667s, 267 tokens) ---
And again, communicate with them. Let them know what your strategic sourcing process looks like. Make that very clear up front so that they understand what they're going to be expected of, right? And I think that those are some of the things that make it truly self-propelling and get that supplier buy-in. It sounds very simple, you know, when I say it, but ultimately these are the things that we believe that so many people miss out on, right, as well. Gustavo, absolutely, I will be able to share the PowerPoint. No worries. And Norman, no, there's absolutely no costs associated with the demo. It's no strings attached. You know, if you would like to get to know a little bit more about our solution, maybe map the maturity of your organization back against, you know, our experts are happy to be able to talk to you, so no worries whatsoever. If there's any other questions, pleas

## 7.Save the chunks

In [7]:
with open("data/chunks.json", "w") as f:
    json.dump(all_chunks, f, indent=2)

print("Saved", len(all_chunks), "chunks to data/chunks.json")

Saved 160 chunks to data/chunks.json


## 8. Install dependencies for Embeddings

In [8]:
!pip install chromadb openai tiktoken


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 9. Setup

In [9]:
!pip install python-dotenv


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
from dotenv import load_dotenv
load_dotenv()

import json
import chromadb
from openai import OpenAI

client = OpenAI()  # now picks up OPENAI_API_KEY from .env
chroma_client = chromadb.PersistentClient(path="./chroma_db")

## 10.Loading chunks

Source: data/chunks.json
Each chunk carries: text, start, end, video_id, title, token_count

In [11]:
with open("data/chunks.json") as f:
    chunks = json.load(f)

print(f"Loaded {len(chunks)} chunks")

Loaded 160 chunks


## 11. Create Chroma collection

In [12]:
collection = chroma_client.get_or_create_collection(
    name="procurement_scm",
    metadata={"hnsw:space": "cosine"}
)

## 12. Embed and upsert chunks

Each chunk gets embedded and stored with its metadata (video_id, title,
timestamps) so retrieved results can be traced back to source + citation.

In [13]:
def embed_text(text: str) -> list[float]:
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=text
    )
    return response.data[0].embedding


batch_size = 50

for i in range(0, len(chunks), batch_size):
    batch = chunks[i:i + batch_size]

    ids = [f"{c['video_id']}_{i+j}" for j, c in enumerate(batch)]
    texts = [c["text"] for c in batch]
    embeddings = [embed_text(t) for t in texts]
    metadatas = [
        {
            "video_id": c["video_id"],
            "title": c["title"],
            "start": c["start"],
            "end": c["end"]
        }
        for c in batch
    ]

    collection.add(
        ids=ids,
        embeddings=embeddings,
        documents=texts,
        metadatas=metadatas
    )

    print(f"Upserted {i + len(batch)}/{len(chunks)}")

print("Done embedding and indexing.")

Upserted 50/160
Upserted 100/160
Upserted 150/160
Upserted 160/160
Done embedding and indexing.


## 13. Sanity check — test a retrieval query

In [14]:
def test_query(query, n_results=3):
    results = collection.query(
        query_embeddings=[embed_text(query)],
        n_results=n_results
    )
    for doc, meta, dist in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0]
    ):
        print(f"--- {meta['title']} ({meta['start']:.0f}s) — dist={dist:.3f} ---")
        print(doc)
        print()

test_query("what are the stages of contract management?")

--- Contract Management in Procurement | Stages & Tools (59s) — dist=0.250 ---
Compliance and risk mitigation ensures regulatory, legal, and internal compliance. Cost control prevents maverick spending and ensures price adherence. Performance tracking monitors supplier adherence to KPIs and SLAs. Dispute resolution provides a formal mechanism to resolve disagreements. 3. Key stages of contract management. Stage 1. Contract creation and negotiation. Identify procurement needs and draft contract terms including deliverables, pricing, and timelines. Collaborate with legal and finance teams for term approvals. Tools used, Microsoft Word for drafting, DocuSign for review workflows, Ironclad for negotiation version tracking. Stage 2. Contract execution. Finalize and sign agreements through digital or physical signatures. Route for internal approvals and record contracts in a central repository. Tools used, Adobe Sign, CLM platforms for tracking execution. Stage 3. Contract monitoring and com

In [15]:
test_query("how do you segment suppliers by importance?")

--- 6-Step Strategic Sourcing Flywheel: Turning Supplier Relationships into Results (1870s) — dist=0.391 ---
From that, well, then you can be able to understand how do you manage that particular supplier in that particular item of your portfolio? What we believe is important is that you're not just focusing on segmenting your suppliers, but that you're automating that segmentation. Technology can help to guide that. But of course, the parameters need to be set by the business together with your organization, in your teams, and of course, aligned with your strategic goals. This automated segmentation, though, is a rule based setup that we do together with our customers. And you should be able to, of course, ascertain this with any technology that you would be finding out there on the market for supplier relationship management. It's a kind of if-then methodology, which is then allowing our customers to quickly segment their supplier bases, approve them, not approve them, but also then b

In [16]:
test_query("what causes stockouts and how is inventory tracked?")

--- Understanding Inventory Management (Inside the Supply Chain Series) Lesson 1 (176s) — dist=0.487 ---
Inventory management can be the heartbeat but of a supply chain or a company and its supply chain, especially because it plays a very important role in optimizing that all the different various aspects of a supply chain and its operations run as smoothly as possible and can actually function. So now we're going to take a look at some of these operations or what is within inventory management. Now look at those next. Now some of the aspects we're going to talk about is balancing supply and demand. I mentioned that a little bit earlier, but we're going to talk about it here too. Inventory management is going to be all about finding the perfect balance between supply and demand. Maintaining the right level of inventory, companies or organizations can ensure that the products are readily available when customers need them. This reduces stockouts, which can lead to lost sales opportuniti

## 13.5 Wrap retrieval as a LangChain Tool

Formalizes the existing retrieval logic as a Tool so it can be handed to
a LangChain agent later, which will decide per-question whether to use
this, a general-knowledge tool, or both.

In [17]:
from langchain.tools import Tool

def retrieve_video_content(query: str) -> str:
    """
    Search the procurement/SCM video corpus and return relevant excerpts
    with source citations. Use this for questions about procurement,
    sourcing, contracts, inventory, demand planning, logistics, risk,
    SRM, sustainability, or AI in procurement.
    """
    results = collection.query(
        query_embeddings=[embed_text(query)],
        n_results=4
    )
    docs = results["documents"][0]
    metas = results["metadatas"][0]

    formatted = "\n\n".join(
        f"[Source: {m['title']} at {m['start']:.0f}s]\n{d}"
        for d, m in zip(docs, metas)
    )
    return formatted


video_retrieval_tool = Tool(
    name="video_content_search",
    func=retrieve_video_content,
    description=(
        "Searches a curated set of procurement and supply chain management "
        "video transcripts. Use this for any question about procurement, "
        "sourcing, contracts, inventory, demand planning, logistics, risk "
        "management, supplier relationships, sustainability, or AI in "
        "procurement. Returns relevant excerpts with source citations."
    )
)

## 13.5.1 Sanity check — confirm the tool works standalone

In [18]:
test_output = video_retrieval_tool.func("what are the stages of contract management?")
print(test_output)

[Source: Contract Management in Procurement | Stages & Tools at 59s]
Compliance and risk mitigation ensures regulatory, legal, and internal compliance. Cost control prevents maverick spending and ensures price adherence. Performance tracking monitors supplier adherence to KPIs and SLAs. Dispute resolution provides a formal mechanism to resolve disagreements. 3. Key stages of contract management. Stage 1. Contract creation and negotiation. Identify procurement needs and draft contract terms including deliverables, pricing, and timelines. Collaborate with legal and finance teams for term approvals. Tools used, Microsoft Word for drafting, DocuSign for review workflows, Ironclad for negotiation version tracking. Stage 2. Contract execution. Finalize and sign agreements through digital or physical signatures. Route for internal approvals and record contracts in a central repository. Tools used, Adobe Sign, CLM platforms for tracking execution. Stage 3. Contract monitoring and compliance. T

## 14 General knowledge Tool

A fallback for questions outside the video corpus — answers from the
LLM's own training knowledge, clearly labeled as such so users can tell
it apart from grounded, cited video-based answers.

In [19]:
def answer_general_knowledge(query: str) -> str:
    """
    Answer a general procurement/SCM knowledge question using the LLM's
    own training data, without retrieving from the video corpus. Used
    when the question falls outside the scope of the indexed videos.
    """
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a procurement and supply chain management "
                    "assistant. Answer using your general knowledge of "
                    "the field. Be clear and concise. Note at the start "
                    "of your answer that this is general knowledge, not "
                    "sourced from the video library."
                )
            },
            {"role": "user", "content": query}
        ],
        temperature=0.3
    )
    return response.choices[0].message.content


general_knowledge_tool = Tool(
    name="general_procurement_knowledge",
    func=answer_general_knowledge,
    description=(
        "Answers general procurement, sourcing, or supply chain "
        "management questions using broad domain knowledge, when the "
        "question is not covered by the indexed video library — for "
        "example, definitions, industry standards, certifications, or "
        "topics the videos don't address. Does not provide source "
        "citations from the videos."
    )
)

## 14.1 Sanity check — confirm the tool works standalone

In [20]:
test_output = general_knowledge_tool.func("What is ethical procurement per CIPS?")
print(test_output)

Ethical procurement, as defined by the Chartered Institute of Procurement & Supply (CIPS), refers to the process of acquiring goods and services in a manner that is responsible, sustainable, and considers the broader social, economic, and environmental impacts. It emphasizes fairness, transparency, and integrity in supplier relationships and decision-making.

Key principles of ethical procurement include:

1. **Sustainability**: Ensuring that procurement practices do not harm the environment and promote sustainable sourcing.
2. **Fair Trade**: Supporting suppliers that adhere to fair labor practices and contribute positively to their communities.
3. **Transparency**: Maintaining open communication and clear processes to build trust among stakeholders.
4. **Accountability**: Taking responsibility for the impact of procurement decisions and ensuring compliance with ethical standards.

Overall, ethical procurement aims to create value not just for the organization but also for society and

## 15. Basic QA chain

Takes a user question → retrieves top-k relevant chunks from Chroma →
passes them as context to an LLM → returns a grounded answer with
source citations (video title + timestamp).

In [21]:
def answer_question(question: str, n_results: int = 4) -> dict:
    """
    Retrieve relevant chunks and generate an answer grounded in them.

    Returns a dict with the answer text and the source chunks used,
    so citations can be shown alongside the response.
    """
    results = collection.query(
        query_embeddings=[embed_text(question)],
        n_results=n_results
    )

    docs = results["documents"][0]
    metas = results["metadatas"][0]

    context = "\n\n".join(
        f"[Source: {m['title']} at {m['start']:.0f}s]\n{d}"
        for d, m in zip(docs, metas)
    )

    system_prompt = (
        "You are a procurement and supply chain management assistant. "
        "Answer the user's question using ONLY the provided context. "
        "If the context doesn't contain enough information to answer, say so. "
        "Cite the video title when referencing specific information."
    )

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"}
        ],
        temperature=0.2
    )

    return {
        "answer": response.choices[0].message.content,
        "sources": [
            {"title": m["title"], "start": m["start"], "video_id": m["video_id"]}
            for m in metas
        ]
    }

## 15.1 Test the full pipeline

In [22]:
result = answer_question("What are the four stages of contract management?")

print("ANSWER:")
print(result["answer"])
print("\nSOURCES:")
for s in result["sources"]:
    print(f"- {s['title']} ({s['start']:.0f}s) — youtube.com/watch?v={s['video_id']}&t={int(s['start'])}")

ANSWER:
The four stages of contract management are:

1. Contract creation and negotiation: Identify procurement needs and draft contract terms including deliverables, pricing, and timelines.
2. Contract execution: Finalize and sign agreements through digital or physical signatures and record contracts in a central repository.
3. Contract monitoring and compliance: Track SLAs, renewal dates, and non-compliance events using dashboards and run audits to check deliverables against timelines.
4. Contract renewal or closure: Evaluate supplier performance and determine whether to extend or terminate the contract, archive closed contracts, and collect post-mortem lessons. 

This information is sourced from the video titled "Contract Management in Procurement | Stages & Tools."

SOURCES:
- Contract Management in Procurement | Stages & Tools (59s) — youtube.com/watch?v=_l-rmPwyv28&t=58
- Contract Management in Procurement | Stages & Tools (129s) — youtube.com/watch?v=_l-rmPwyv28&t=129
- Contract

## 16. Build the agent

Combines both tools with a decision-making LLM. The agent reads the
user's question, compares it against each tool's description, and
decides which to call — video retrieval, general knowledge, or both.

In [23]:
!pip install langchain-openai


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 16.1Enable langSmith Tracing

In [24]:
import os

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "procurement-scm-assistant"

In [25]:
from langchain.agents import create_tool_calling_agent, AgentExecutor
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

tools = [video_retrieval_tool, general_knowledge_tool]

agent_system_prompt = agent_system_prompt = (
    "You are a procurement and supply chain management assistant. "
    "You have two tools available:\n"
    "- video_content_search: searches a curated video library covering "
    "procurement, sourcing, contracts, inventory, demand planning, "
    "logistics, risk, SRM, sustainability, AI in procurement\n"
    "- general_procurement_knowledge: general domain knowledge, not "
    "sourced from videos\n\n"
    "MANDATORY RULE: For every question, you MUST call "
    "video_content_search FIRST, before considering any other tool. "
    "Only call general_procurement_knowledge if video_content_search's "
    "results are empty, clearly irrelevant, or explicitly insufficient "
    "to answer the question. Never skip video_content_search."
)

prompt = ChatPromptTemplate.from_messages([
    ("system", agent_system_prompt),
    MessagesPlaceholder("chat_history", optional=True),
    ("human", "{input}"),
    MessagesPlaceholder("agent_scratchpad"),
])

agent = create_tool_calling_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

## 16.2 Sanity check — test routing on both question types

In [26]:
# Should route to video_content_search
result1 = agent_executor.invoke({"input": "What are the stages of contract management?"})
print(result1["output"])

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")



Invoking: `video_content_search` with `stages of contract management`


[Source: Contract Management in Procurement | Stages & Tools at 59s]
Compliance and risk mitigation ensures regulatory, legal, and internal compliance. Cost control prevents maverick spending and ensures price adherence. Performance tracking monitors supplier adherence to KPIs and SLAs. Dispute resolution provides a formal mechanism to resolve disagreements. 3. Key stages of contract management. Stage 1. Contract creation and negotiation. Identify procurement needs and draft contract terms including deliverables, pricing, and timelines. Collaborate with legal and finance teams for term approvals. Tools used, Microsoft Word for drafting, DocuSign for review workflows, Ironclad for negotiation version tracking. Stage 2. Contract execution. Finalize and sign agreements through digital or physical signatures. Route for internal approvals and record contracts in a central repository. Tools used, Adobe Sign, CLM platfor

In [27]:
# Should route to general_procurement_knowledge
result2 = agent_executor.invoke({"input": "What is ethical procurement per CIPS?"})
print(result2["output"])

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")



Invoking: `video_content_search` with `ethical procurement CIPS`


[Source: Sustainable Procurement Explained: Key Frameworks and Benefits at 271s]
benefits and positive impacts that sustainable procurement can have on business including cost savings risk reduction and brand reputation enhancement we are excited to delve into these crucial aspects of sustainable procurement and equip you with valuable insights to drive positive change within your organization let's get started every business needs to change to become more sustainable for starters sustainable procurement refers to the approach of adopting environmental social and governmental factors while also considering the price and quality of materials that an organization will acquire sustainability is becoming increasingly mainstream every organization must contend with the transition to for example renewable energy every business must be resilient to climate change and shocks and even more importantly every business needs to ea

## 17 Sanity check — confirm traces are logging in Langsmith

In [28]:
result = agent_executor.invoke({"input": "What are the four components of a procurement contract?"})
print(result["output"])
print("\nCheck smith.langchain.com under project 'procurement-scm-assistant' to see this trace.")

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")



Invoking: `video_content_search` with `components of a procurement contract`


[Source: Contract Management in Procurement | Stages & Tools at 129s]
Stage 4. Contract renewal or closure. Evaluate supplier performance and determine whether to extend or terminate the contract. Archive closed contracts and collect postmorm lessons. Example, a software company uses a 12-month SaaS subscription for a CRM tool. Near expiration, procurement evaluates usage, cost effectiveness, and vendor support to decide on renewal. Four, components of a procurement contract. Scope of work, SOW clear definition of the goods or services to be delivered. Pricing and payment terms include structure, fixed, milestone, hourly, due dates, and penalties. Service level agreements, SLAs, and KPIs, define expectations for delivery times, quality, and issue resolution. Termination clauses, outline under what circumstances the contract may be terminated early. Force majeure, provision to manage unforeseen disruptions, 

## 18. Add conversational memory to the agent

Wraps agent_executor calls with a running chat history so follow-up
questions ("what about the second one?") resolve correctly without the
user re-stating context. Uses LangChain's message objects to keep the
history in the format the agent's prompt template expects.

In [29]:
from langchain_core.messages import HumanMessage, AIMessage

chat_history = []

def ask_agent(question: str) -> dict:
    """
    Invoke the agent with the running chat history, then update history
    with this turn's question and answer.
    """
    result = agent_executor.invoke({
        "input": question,
        "chat_history": chat_history
    })

    chat_history.append(HumanMessage(content=question))
    chat_history.append(AIMessage(content=result["output"]))

    return result

## 18.1 Sanity check — test memory with a follow-up

In [30]:
r1 = ask_agent("What are the four stages of contract management?")
print(r1["output"])

print("\n---\n")



Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")



Invoking: `video_content_search` with `four stages of contract management`


[Source: Contract Management in Procurement | Stages & Tools at 59s]
Compliance and risk mitigation ensures regulatory, legal, and internal compliance. Cost control prevents maverick spending and ensures price adherence. Performance tracking monitors supplier adherence to KPIs and SLAs. Dispute resolution provides a formal mechanism to resolve disagreements. 3. Key stages of contract management. Stage 1. Contract creation and negotiation. Identify procurement needs and draft contract terms including deliverables, pricing, and timelines. Collaborate with legal and finance teams for term approvals. Tools used, Microsoft Word for drafting, DocuSign for review workflows, Ironclad for negotiation version tracking. Stage 2. Contract execution. Finalize and sign agreements through digital or physical signatures. Route for internal approvals and record contracts in a central repository. Tools used, Adobe Sign, CLM pl

In [31]:
r2 = ask_agent("Can you explain the second one in more detail?")
print(r2["output"])

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")



Invoking: `video_content_search` with `contract execution in contract management`


[Source: Contract Management in Procurement | Stages & Tools at 59s]
Compliance and risk mitigation ensures regulatory, legal, and internal compliance. Cost control prevents maverick spending and ensures price adherence. Performance tracking monitors supplier adherence to KPIs and SLAs. Dispute resolution provides a formal mechanism to resolve disagreements. 3. Key stages of contract management. Stage 1. Contract creation and negotiation. Identify procurement needs and draft contract terms including deliverables, pricing, and timelines. Collaborate with legal and finance teams for term approvals. Tools used, Microsoft Word for drafting, DocuSign for review workflows, Ironclad for negotiation version tracking. Stage 2. Contract execution. Finalize and sign agreements through digital or physical signatures. Route for internal approvals and record contracts in a central repository. Tools used, Adobe Sign,

## 19. Video metadata Tool

Answers questions about the video library itself — what's covered,
how many videos, which titles exist — without doing semantic search
over chunk content. Useful for questions like "what topics do you
cover?" or "how many videos are in your library?"

In [32]:
def get_video_metadata(query: str) -> str:
    """
    Answer questions about the video library's contents — titles, topic
    coverage, video count — without searching chunk text. Builds its
    answer directly from the loaded transcript metadata.
    """
    video_list = []
    for vid, content in data.items():
        video_list.append(f"- {content['title']} (video_id: {vid})")

    summary = (
        f"The library contains {len(data)} videos:\n"
        + "\n".join(video_list)
    )
    return summary


video_metadata_tool = Tool(
    name="video_library_metadata",
    func=get_video_metadata,
    description=(
        "Answers questions about the video library's contents as a "
        "whole — for example, how many videos are indexed, what topics "
        "or titles are covered, or which video discusses a specific "
        "subject at a high level. Does NOT search inside video "
        "transcripts — use video_content_search for that instead."
    )
)

## 19.1 Sanity check — confirm the tool works standalone

In [33]:
test_output = video_metadata_tool.func("what topics are covered?")
print(test_output)

The library contains 10 videos:
- 6-Step Strategic Sourcing Flywheel: Turning Supplier Relationships into Results (video_id: uSjrTpJHn2g)
- Contract Management in Procurement | Stages & Tools (video_id: _l-rmPwyv28)
- Demand Planning Explained | Process, Benefits & Forecasting (video_id: W10PJUFTH9M)
- An Introduction to Logistics and Supply Chain Management (video_id: GCK18JyPVXI)
- Digital Transformation & AI in Procurement (video_id: R297gX6-KvQ)
- Sustainable Procurement Explained: Key Frameworks and Benefits (video_id: U1E6rtnreoA)
- What is Supplier Relationship Management? Supply Chain 101 (video_id: dswoMsZwRuo)
- Supply Chain Risk Management Strategy | Response to Risk | Contingency Plan (video_id: ymFVfng3dXE)
- What is Procurement? Procurement Process Explained in 12 minutes (video_id: ZCIJQJRw6xw)
- Understanding Inventory Management (Inside the Supply Chain Series) Lesson 1 (video_id: 0ZDrpf5aMiw)


## 19.2 Add the tool to the agent

In [34]:
tools = [video_retrieval_tool, general_knowledge_tool, video_metadata_tool]

agent_system_prompt = (
    "You are a procurement and supply chain management assistant. "
    "You have three tools available:\n"
    "- video_content_search: searches inside the curated video library "
    "for specific procurement/SCM content\n"
    "- video_library_metadata: answers questions about the library "
    "itself — titles, topic coverage, video count\n"
    "- general_procurement_knowledge: general domain knowledge, not "
    "sourced from videos\n\n"
    "Use video_library_metadata for questions about what the library "
    "contains or covers, not for questions seeking specific content "
    "from within a video. For content questions, MANDATORY RULE: call "
    "video_content_search FIRST. Only use general_procurement_knowledge "
    "if video_content_search's results are empty, irrelevant, or "
    "insufficient."
)

prompt = ChatPromptTemplate.from_messages([
    ("system", agent_system_prompt),
    MessagesPlaceholder("chat_history", optional=True),
    ("human", "{input}"),
    MessagesPlaceholder("agent_scratchpad"),
])

agent = create_tool_calling_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

## 19.3 Sanity check — test metadata routing through the agent

In [35]:
result = ask_agent("How many videos are in your library, and what topics do they cover?")
print(result["output"])

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")



Invoking: `video_library_metadata` with `video count and topics covered`


The library contains 10 videos:
- 6-Step Strategic Sourcing Flywheel: Turning Supplier Relationships into Results (video_id: uSjrTpJHn2g)
- Contract Management in Procurement | Stages & Tools (video_id: _l-rmPwyv28)
- Demand Planning Explained | Process, Benefits & Forecasting (video_id: W10PJUFTH9M)
- An Introduction to Logistics and Supply Chain Management (video_id: GCK18JyPVXI)
- Digital Transformation & AI in Procurement (video_id: R297gX6-KvQ)
- Sustainable Procurement Explained: Key Frameworks and Benefits (video_id: U1E6rtnreoA)
- What is Supplier Relationship Management? Supply Chain 101 (video_id: dswoMsZwRuo)
- Supply Chain Risk Management Strategy | Response to Risk | Contingency Plan (video_id: ymFVfng3dXE)
- What is Procurement? Procurement Process Explained in 12 minutes (video_id: ZCIJQJRw6xw)
- Understanding Inventory Management (Inside the Supply Chain Series) Lesson 1 (video_id: 0ZDrpf5aMiw)T

## 20. Adding a voice input tool

In [36]:
pip install openai sounddevice scipy


   ------------- -------------------------- 1/3 [cffi]
   -------------------------- ------------- 2/3 [sounddevice]
   ---------------------------------------- 3/3 [sounddevice]

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [37]:
def transcribe_audio(audio_file) -> str:
    """Transcribe an audio file to text using OpenAI Whisper API."""
    transcript = client.audio.transcriptions.create(
        model="whisper-1",
        file=audio_file
    )
    return transcript.text

## 20.1 Voice input sanity check

In [40]:
import sounddevice as sd
from scipy.io.wavfile import write

fs = 44100
duration = 5  # seconds
print("Recording...")
recording = sd.rec(int(duration * fs), samplerate=fs, channels=1)
sd.wait()
write("test_voice.wav", fs, recording)
print("Done recording.")

text = transcribe_audio(open("test_voice.wav", "rb"))
print("Transcribed text:", text)

result = ask_agent(text)
print(result["output"])

Recording...
Done recording.


Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


Transcribed text: Define the procurement process.

Invoking: `general_procurement_knowledge` with `Define the procurement process.`


The procurement process is a systematic approach to acquiring goods and services needed by an organization. It typically involves several key steps:

1. **Needs Identification**: Determine what goods or services are required, including specifications and quantities.

2. **Supplier Research**: Identify potential suppliers who can provide the required goods or services.

3. **Request for Proposal (RFP)/Quotation (RFQ)**: Solicit bids or proposals from suppliers to understand pricing and terms.

4. **Supplier Evaluation**: Assess suppliers based on criteria such as price, quality, reliability, and service.

5. **Negotiation**: Discuss terms and conditions with selected suppliers to reach a mutually beneficial agreement.

6. **Purchase Order (PO) Issuance**: Officially place an order with the chosen supplier, detailing the terms of the purchase.

7. **Order 

In [41]:
test_queries = [
    "What does the video about contract management say, and how many videos do you have total?",  # forces multi-tool use
    "Tell me about supplier risk management",  # could hit RAG content tool OR general_knowledge — ambiguous on purpose
    "What did I just ask you?",  # should hit memory, not RAG or metadata
    "How many videos discuss sustainability, and briefly explain what sustainable procurement means",  # metadata + general knowledge combo
    "What's the capital of France?",  # should NOT match any procurement tool — tests fallback/refusal behavior
    "Summarize everything you know about demand planning from the video library",  # should hit RAG, not general_knowledge, since "from the video library" is explicit
]

for q in test_queries:
    print(f"\n{'='*60}\nQUERY: {q}\n{'='*60}")
    result = ask_agent(q)
    print(result["output"])

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")



QUERY: What does the video about contract management say, and how many videos do you have total?

Invoking: `video_content_search` with `contract management`


[Source: Contract Management in Procurement | Stages & Tools at 59s]
Compliance and risk mitigation ensures regulatory, legal, and internal compliance. Cost control prevents maverick spending and ensures price adherence. Performance tracking monitors supplier adherence to KPIs and SLAs. Dispute resolution provides a formal mechanism to resolve disagreements. 3. Key stages of contract management. Stage 1. Contract creation and negotiation. Identify procurement needs and draft contract terms including deliverables, pricing, and timelines. Collaborate with legal and finance teams for term approvals. Tools used, Microsoft Word for drafting, DocuSign for review workflows, Ironclad for negotiation version tracking. Stage 2. Contract execution. Finalize and sign agreements through digital or physical signatures. Route for internal app

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


The video titled **"Contract Management in Procurement | Stages & Tools"** discusses the following key points:

1. **Importance of Compliance and Risk Mitigation**: It emphasizes the need for regulatory, legal, and internal compliance, cost control to prevent maverick spending, performance tracking to monitor supplier adherence to KPIs and SLAs, and a formal mechanism for dispute resolution.

2. **Key Stages of Contract Management**:
   - **Contract Creation and Negotiation**: Identifying procurement needs, drafting terms, and collaborating with legal and finance teams.
   - **Contract Execution**: Finalizing and signing agreements, routing for internal approvals, and recording contracts in a central repository.
   - **Contract Monitoring and Compliance**: Tracking SLAs, renewal dates, and conducting audits to ensure compliance.
   - **Contract Renewal or Closure**: Evaluating supplier performance to decide on contract extensions or terminations.

3. **Components of a Procurement Contr

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


...

You asked about **supplier risk management** and requested information regarding its components and importance in procurement and supply chain management.

> Finished chain.
You asked about **supplier risk management** and requested information regarding its components and importance in procurement and supply chain management.

QUERY: How many videos discuss sustainability, and briefly explain what sustainable procurement means

Invoking: `video_content_search` with `sustainability`


[Source: Sustainable Procurement Explained: Key Frameworks and Benefits at 1005s]
transition to sustainability as a challenge we need to start looking at it as an opportunity for innovation by using sustainable procurement to stimulate innovation from the supply chains we can gain greater shared value and generate new markets now to make this easy to understand we have categorized the benefits into three categories economic environmental and social benefits firstly let's discuss the economic benefits numb

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


There is **1 video** in the library that specifically discusses sustainability, titled **"Sustainable Procurement Explained: Key Frameworks and Benefits."**

### Sustainable Procurement Definition:
Sustainable procurement refers to the approach of integrating environmental, social, and governance (ESG) factors into the procurement process while also considering price and quality. It aims to identify and reduce the negative environmental and social impacts associated with the supply chain. 

Key aspects of sustainable procurement include:

- **Life Cycle Consideration**: Evaluating the entire life cycle of products and services, from sourcing raw materials to disposal, to minimize environmental impact.
- **Social Responsibility**: Ensuring that procurement practices respect human rights and promote fair labor practices throughout the supply chain.
- **Economic Viability**: Balancing sustainability with cost-effectiveness to ensure that procurement decisions support the organization's fi

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


The capital of France is Paris.

> Finished chain.
The capital of France is Paris.

QUERY: Summarize everything you know about demand planning from the video library

Invoking: `video_content_search` with `demand planning`


[Source: Demand Planning Explained | Process, Benefits & Forecasting at 85s]
in the market. Demand planning process typically involves multiple elements like first demand planner uses past or historical sales data to create a statistical forecast. Then demand planners works with the customers, product managers to override the statistical forecast by considering their inputs if they anticipate greater or lower demand and by how much and then demand planner finalize consensus forecast, which is also called as unconstrained consensus demand forecast because at this level there is no constraint in the demand plan so far. Now why to do demand planning? So let's take an example. So example here is packet of bread which has lead time of three days, which is basically proc